# §13.1.5 — 최소 구현과 순열 등변성의 수치 검증

> 딥러닝 교재 · 3부 13장 1절 5항 (🐍)
> 선행: §13.1.1(정의) · §13.1.2(순열 등변성) · §13.1.3(커널 평활화 독법)

## 이 노트북이 답하는 질문

1. **정리 13.1.1의 등변성은 수치로 성립하는가?** 머신 정밀도 수준인지 확인한다.
2. **위치 부호화를 더하면 등변성은 어떻게 되는가?** 깨진다 — 깨라고 넣는 것이다.
3. **어텐션은 정말 나다라야–왓슨 평활기인가?** 1차원 회귀를 어텐션 함수 그대로로 수행한다.

**예상 실행 시간** CPU 약 20초.

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 최소 구현 — 본문 마진 리스팅 그대로

수치 안정화(행 최댓값 빼기, §2.3.5)까지 포함해도 몇 줄이다.

In [ ]:
def self_attention(X, Wq, Wk, Wv, scale=None):
    # X: (T,d). 반환: 출력 Y와 어텐션 행렬 A
    d = X.shape[1]
    scale = np.sqrt(d) if scale is None else scale
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    E = Q @ K.T / scale
    E -= E.max(1, keepdims=True)
    A = np.exp(E)
    A /= A.sum(1, keepdims=True)
    return A @ V, A

d, T = 16, 12
rn = np.random.default_rng(SEED)
Wq, Wk, Wv = (rn.standard_normal((d, d)) / np.sqrt(d) for _ in range(3))
X = rn.standard_normal((T, d))
Y, A = self_attention(X, Wq, Wk, Wv)
print(f"출력 {Y.shape}, 어텐션 행렬 {A.shape}, 행 합 = {A.sum(1).round(12)[:4]} ...")

---
## 2. 순열 등변성 — 정리 13.1.2의 수치 검증

무작위 순열 $P$ 여러 개에 대해 $\|\operatorname{SA}(PX)-P\,\operatorname{SA}(X)\|$를 잰다.
그다음 사인 위치 부호화(식 13.4.2)를 **더한 뒤** 같은 것을 잰다.

In [ ]:
def sin_pe(T, d):
    pos = np.arange(T)[:, None]
    l2 = np.arange(0, d, 2)[None, :]
    ang = pos / (10000.0 ** (l2 / d))
    pe = np.zeros((T, d))
    pe[:, 0::2] = np.sin(ang); pe[:, 1::2] = np.cos(ang)
    return pe

N_PERM = 30
errs_plain, errs_pe = [], []
PE = sin_pe(T, d)
for _ in range(N_PERM):
    perm = rn.permutation(T)
    # 위치 부호화 없음: SA(PX) 와 P·SA(X)
    Y1, _ = self_attention(X[perm], Wq, Wk, Wv)
    errs_plain.append(np.abs(Y1 - Y[perm]).max())
    # 위치 부호화 있음: SA(P X + PE) 와 P·SA(X + PE)
    Yp, _ = self_attention(X + PE, Wq, Wk, Wv)
    Y2, _ = self_attention(X[perm] + PE, Wq, Wk, Wv)
    errs_pe.append(np.abs(Y2 - Yp[perm]).max())
print(f"등변성 오차 (부호화 없음): 최대 {max(errs_plain):.2e}  ← 부동소수점 반올림 수준")
print(f"등변성 오차 (사인 부호화): 최대 {max(errs_pe):.2e}  ← 자릿수 단위로 깨짐")

---
## 3. 나다라야–왓슨 회귀 — 어텐션 함수 그대로

표본 $(x_j, y_j)$에서 $\hat m(x)=\sum_j \alpha_j y_j$, $\alpha=\operatorname{softmax}(\text{유사도}/\tau)$.
내적 커널이 **근접 커널**이 되도록 스칼라 입력을 원 위에 임베딩한다:
$\varphi(x)=(\cos\theta,\sin\theta)$, $\theta\propto x$. 그러면
$\varphi(x)^\top\varphi(x')=\cos(\theta-\theta')$ — 가까울수록 큰 점수다.
(위치를 원 위의 회전으로 적는 §13.4의 사인·RoPE 부호화와 같은 발상이다.)

In [ ]:
def ring_embed(x, x_lo, x_hi):
    th = (x - x_lo) / (x_hi - x_lo) * np.pi * 1.6   # 0..1.6π — 감김 없이
    return np.stack([np.cos(th), np.sin(th)], axis=-1)

n_tr = 40
x_tr = np.sort(rn.uniform(-1, 1, n_tr))
y_tr = np.sin(3 * x_tr) + 0.25 * rn.standard_normal(n_tr)
x_gr = np.linspace(-1, 1, 200)
K_tr = ring_embed(x_tr, -1, 1)
Q_gr = ring_embed(x_gr, -1, 1)

def nw_attention(Q, K, y, tau):
    E = Q @ K.T / tau
    E -= E.max(1, keepdims=True)
    A = np.exp(E); A /= A.sum(1, keepdims=True)
    return A @ y, A

TAUS = [0.002, 0.02, 0.5]
fits = {tau: nw_attention(Q_gr, K_tr, y_tr, tau)[0] for tau in TAUS}
for tau in TAUS:
    resid = fits[tau] - np.sin(3 * x_gr)
    print(f"tau={tau:5g}: 참 함수 대비 RMS 오차 {np.sqrt((resid**2).mean()):.3f}")

---
## 4. 교재 그림 — fig_13_1_5

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) 어텐션 행렬
ax = axes[0]
imv = ax.imshow(A, cmap='viridis', vmin=0)
plt.colorbar(imv, ax=ax, fraction=0.046)
ax.grid(False)
ax.set_xlabel(lab('키 위치 $j$', 'key $j$'))
ax.set_ylabel(lab('질의 위치 $i$', 'query $i$'))
ax.set_title(lab('(a) 어텐션 행렬 $\\alpha$ — 각 행의 합이 1', '(a) attention matrix'), fontsize=10)

# (b) 등변성 오차
ax = axes[1]
xs = np.arange(N_PERM)
ax.semilogy(xs, errs_plain, 'o', color=CB[5], ms=4, label=lab('위치 부호화 없음', 'no PE'))
ax.semilogy(xs, errs_pe, 's', color=CB[4], ms=4, label=lab('사인 부호화 추가', 'with PE'))
ax.axhline(np.finfo(float).eps, color='k', lw=0.7, ls=':')
ax.text(0, np.finfo(float).eps * 1.8, lab('머신 엡실론', 'machine eps'), fontsize=8)
ax.set_xlabel(lab('무작위 순열 시행', 'random permutation'))
ax.set_ylabel(lab('$\\|\\mathrm{SA}(PX)-P\\,\\mathrm{SA}(X)\\|_\\infty$', 'equivariance error'))
ax.set_title(lab('(b) 등변성 — 그리고 그것을 깨는 유일한 것', '(b) permutation equivariance'), fontsize=10)
ax.legend(fontsize=8, loc='center right')

# (c) NW 회귀
ax = axes[2]
ax.plot(x_tr, y_tr, 'o', color='0.6', ms=4, label=lab('표본', 'data'))
ax.plot(x_gr, np.sin(3 * x_gr), '-', color='k', lw=1, label=lab('참 함수', 'true'))
ax.plot(x_gr, fits[0.02], '-', color=CB[5], lw=1.8, label=lab('어텐션 평활 ($\\tau=0.02$)', 'attention'))
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(lab('(c) 어텐션 = 나다라야–왓슨 평활기', '(c) NW regression'), fontsize=10)
ax.legend(fontsize=8)

# (d) 온도(스케일)와 평활의 정도
ax = axes[3]
for i, tau in enumerate(TAUS):
    ax.plot(x_gr, fits[tau], '-', color=CB[i + 1], lw=1.5, label=f'$\\tau={tau}$')
ax.plot(x_tr, y_tr, 'o', color='0.75', ms=3, zorder=0)
ax.set_xlabel('$x$'); ax.set_ylabel('$y$')
ax.set_title(lab('(d) 스케일은 커널의 대역폭 — 과소·과대 평활', '(d) bandwidth'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_13_1_5')
plt.show()

> ### 읽는 법
>
> (b) 위치 부호화가 없으면 등변성 오차는 머신 엡실론 수준 — 정리 13.1.2가 부동소수점의
> 한계까지 정확히 성립한다. 사인 부호화를 더하는 순간 오차는 $10^{0}$ 규모로 뛴다.
> **순서에 대한 민감성은 전적으로 위치 부호화가 만든다**(§13.1.7).
> (c)–(d) 사영을 항등으로 둔 어텐션은 문자 그대로 나다라야–왓슨 평활기다. 스케일
> $\tau$가 커널 대역폭이어서, 작으면 표본을 좇는 과소 평활(최근접 복사), 크면 전체
> 평균으로 뭉개지는 과대 평활이 된다. §13.2의 $\sqrt d$는 이 대역폭을 초기화 시점에
> 안전한 값으로 놓는 장치다.

---
## 5. 자기 점검

1. (b)에서 부호화 있는 쪽의 오차가 $10^{-1}$–$10^{0}$ 규모인 이유를 생각해 보라. 오차의 크기는 무엇에 비례하겠는가?
2. 인과 마스크(§13.5)를 추가하면 등변성은 어떻게 되는가? 코드를 고쳐 확인하라.
3. (d)에서 $\tau\to0$ 극한의 평활기는 무엇이 되는가? §2.4.3의 argmax 완화와 연결하라.
4. `ring_embed`의 각도 범위를 $2\pi$ 이상으로 키우면 회귀가 어떻게 망가지는가? RoPE의 파장 이야기(§13.4.5)와 연결하라.

## 6. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `TAUS` | 3절 | [0.002, 0.02, 0.5] | 대역폭 사다리 |
| `N_PERM` | 2절 | 30 | 검증 시행 수 |
| `n_tr` | 3절 | 40 | 표본 밀도와 평활의 관계 |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")